> **Public copy.** The outputs below are from the original run on the private database. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E; releases such as 1.51 and "Vendor C 3.0"), so every number is exactly as computed. The database itself is not published; the anonymised data files are in `data/`.


# 07 The fact table: one row per component per release

**Goal:** build the table the model learns from (the course mentor's advice: one row = one component in one release; never model from a rolled-up release table). Each row is one **component** (for example Payments and PED) in one **release** (for example 1.52), with two kinds of columns:
- **What happened** in that release: defects found, how many were Critical or High, priority-weighted defects, reopened defects, and defects per 10 person-days. These are used only for the label and the dashboard, never as model inputs.
- **What was known before the release started**: the model's inputs.

**Rules carried over from earlier notebooks**
- Only defects that count for risk (notebook 01) and come from release testing, not upgrade work (notebook 04).
- A defect counts fully in every component it touches (main and other components, notebook 05). Release totals count each defect once.
- The 52 releases in the model and the split from notebook 06 (train = started before 1 Jan 2025). 1.37 and Vendor C 3.0 have estimated effort and are flagged, so the model step can run with and without them.

**Inputs chosen by Jagan (24 Nov 2026):** priority weights Critical 8, High 4, Medium 2, Low 1 (each level doubles, so blockers get strong focus; priority works as severity in this team), and recency half-lives of 2 and 4 releases (both are built; the model step keeps whichever predicts better).

The feature code lives in `scripts/hs_features.py`, so the model notebook can score the next release with exactly the same rules.

In [1]:
import sqlite3, sys
from pathlib import Path
import numpy as np
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
print('Reading from:', CLEAN)
from hs_features import load_context, history, FEATURES_NUM, PRIORITY_WEIGHTS, HALF_LIVES

with sqlite3.connect(CLEAN) as cc:
    ctx = load_context(cc)
timeline, links = ctx.timeline, ctx.links
print('Priority weights:', PRIORITY_WEIGHTS, '| half-lives (releases):', HALF_LIVES)
print(len(ctx.counted), 'risk defects from release testing;', len(ctx.in_release), 'of them have a release')
print(len(links), 'defect-component links,', links.key.nunique(), 'defects')

Reading from: data/private/hindsight_clean.sqlite
Priority weights: {'Critical': 8, 'High': 4, 'Medium': 2, 'Low': 1} | half-lives (releases): (2, 4)
2475 risk defects from release testing; 2053 of them have a release
3205 defect-component links, 2033 defects


## 1. Which components get a row in each stream

For each vendor stream, the components that had defects in at least **two training releases** of that stream. A component with no defects in a release still gets a row (zero defects), because "no trouble" is also an answer the model must learn.

Vendor D's own releases (Vendor D) have no training release, so each gets one row for the Vendor D features component. Most Vendor D defects are recorded inside the Vendor A release they shipped in (for example 1.50), where the Vendor D features component has its own row.

In [2]:
model = timeline[timeline.in_model].copy()
train_links = links[links.split == 'train']
universe = (train_links.groupby(['vendor', 'name']).release_key.nunique()
            .reset_index(name='train_releases'))
universe = universe[universe.train_releases >= 2][['vendor', 'name']]
for vendor, comps in {'Vendor D': ['Vendor D features']}.items():        # streams with no training release
    if vendor not in set(universe.vendor):
        universe = pd.concat([universe, pd.DataFrame({'vendor': vendor, 'name': comps})], ignore_index=True)
print(universe.groupby('vendor').name.apply(lambda s: ', '.join(sorted(s))).to_string())

vendor
Vendor A    Assortment, Reporting, Vendor D, ...
Vendor B    Reporting, Environment and inf...
Vendor C    Reporting, Environment and inf...
Vendor D                                                Vendor D
Vendor E    Vendor E, Promotions, pricing and loyalty, Recei...


In [3]:
in_model_links = links[links.in_model.fillna(False).astype(bool)]
outside = in_model_links.merge(universe, on=['vendor', 'name'], how='left', indicator=True)
outside = outside[outside._merge == 'left_only']
print(f'{len(outside)} of {len(in_model_links)} defect-component links in model releases fall outside the rows '
      '(a component new to that stream). They still appear in the dashboard:')
print(outside.groupby(['vendor', 'name']).size().to_string())

18 of 2941 defect-component links in model releases fall outside the rows (a component new to that stream). They still appear in the dashboard:
vendor    name                          
Vendor A  DRS (deposit return)              4
Vendor B  Assortment                        2
          Vendor D                             5
          Vendor E                            1
Vendor C  EPP                               1
          Gift cards                        1
Vendor D  Self-checkout (SCO)               1
Vendor E  Environment and infrastructure    3


## 2. What happened: defects per component and release

In [4]:
rows = model[['release_key', 'vendor', 'release_stream', 'release_order', 'window_start', 'window_end',
               'person_days', 'effort_estimated', 'split']].merge(universe, on='vendor')
rows = rows.rename(columns={'name': 'component'})

per = (links.groupby(['release_key', 'name'])
       .agg(defects=('key', 'nunique'), defects_main=('main', 'sum'), critical_high=('critical_high', 'sum'),
            priority_weighted=('priority_weight', 'sum'), reopened=('reopened', 'sum'), date_only=('date_only', 'sum'))
       .reset_index().rename(columns={'name': 'component'}))
fact = rows.merge(per, on=['release_key', 'component'], how='left')
for col in ['defects', 'defects_main', 'critical_high', 'priority_weighted', 'reopened', 'date_only']:
    fact[col] = fact[col].fillna(0).astype(int)

# Per 10 person-days of the release's testing (all runs; estimated for 1.37 and Vendor C 3.0)
fact['density_per_10pd'] = (fact.defects / fact.person_days * 10).round(3)
fact['sev_density_per_10pd'] = (fact.priority_weighted / fact.person_days * 10).round(3)
print(len(fact), 'rows;', int((fact.defects > 0).sum()), 'with defects;', int((fact.defects == 0).sum()), 'with none')
fact.groupby('split')[['defects', 'critical_high', 'priority_weighted', 'reopened']].sum()

541 rows; 322 with defects; 219 with none


,defects,critical_high,priority_weighted,reopened
split,,,,
test,442,123,1169,7
train,2481,1233,8249,56


## 3. What was known before the release started: the model's inputs

"History" means releases **of the same stream that had finished before this release started**. Releases still running at that moment are not used, so the model never sees the future. The old releases 1.28 to 1.36 count in the history (Jagan's decision), but only releases with effort data count in the per-person-day measures.

| Feature | the course mentor's measure | Meaning |
| --- | --- | --- |
| hist_rate_per_10pd | Defect density | The component's defects per 10 person-days over all earlier releases: its **usual rate** |
| prev_density_per_10pd | Defect density | The same, in the most recent finished release |
| recent_rate_hl2, recent_rate_hl4 | Recency (half-life) | The usual rate with recent releases counting more: a release's weight halves every 2 (or 4) releases |
| hist_sev_rate_per_10pd | Severity-weighted defects | Earlier defects weighted Critical 8, High 4, Medium 2, Low 1, per 10 person-days |
| spread_releases, spread_share | Spread | In how many earlier releases (and what share of them) the component had defects |
| hist_reopened | Fix churn | Earlier defects of the component that were reopened (the fix did not hold) |
| hist_share | Context | The component's share of all the stream's earlier defects |
| recent_90d | Context | Its defects in any release in the 90 days before this release started |
| n_prev_releases, gap_days | Context | How many earlier releases the stream has, and days since the last one finished |
| n_concurrent | Context | Other releases running when this one started (team load) |
| shared_vendor_work | Context | The tracker shows shared work with another vendor in this release (for example Vendor D inside 1.50) |

In [5]:
feats = fact.apply(lambda r: pd.Series(history(ctx, r.vendor, r.window_start, r.component, r.release_key)), axis=1)
fact = pd.concat([fact, feats], axis=1)
example = fact[(fact.release_key == '1.52') & (fact.component == 'Payments and PED')]
example[['release_key', 'component', 'defects', 'density_per_10pd'] + FEATURES_NUM].T

,448
release_key,1.52
component,Payments and PED
defects,52
density_per_10pd,2.543
hist_rate_per_10pd,0.850952
prev_density_per_10pd,0.138889
recent_rate_hl2,0.49387
recent_rate_hl4,0.669557
hist_sev_rate_per_10pd,2.991425
spread_releases,18.0


**Missing history.** Rows whose stream has no earlier finished release with effort data have empty per-person-day features (for example the first Vendor B release, CI 2). They stay empty here; the model step fills them from the training data only.

In [6]:
fact[FEATURES_NUM].isna().sum()[lambda s: s > 0].to_frame('empty values')

,empty values
hist_rate_per_10pd,61
prev_density_per_10pd,61
recent_rate_hl2,61
recent_rate_hl4,61
hist_sev_rate_per_10pd,61
spread_share,26
hist_share,29
gap_days,26


In [7]:
fact.groupby('split')[FEATURES_NUM].mean().T.round(2)

split,test,train
hist_rate_per_10pd,0.64,0.86
prev_density_per_10pd,0.92,0.71
recent_rate_hl2,0.76,0.84
recent_rate_hl4,0.73,0.87
hist_sev_rate_per_10pd,2.04,2.51
spread_releases,9.54,4.61
spread_share,0.56,0.54
hist_reopened,1.60,0.76
hist_share,0.13,0.13
recent_90d,7.27,5.54


## 4. Checks

1. The release totals must match notebook 06 (risk defects per release).
2. No component count may be larger than its release total.
3. No history feature may use a release that had not finished when the row's release started.
4. Each recency feature is a weighted average, so it must sit between the lowest and highest earlier density.
5. Priority-weighted defects must lie between 1 and 8 times the defect count.

In [8]:
totals = ctx.in_release.groupby('release_key').key.nunique()
check = model.set_index('release_key').risk_defects
assert (totals.reindex(check.index).fillna(0).astype(int) == check).all(), 'release totals differ from notebook 06'
print('1. Release totals match notebook 06 for all', len(check), 'releases in the model')

over = fact.defects > fact.release_key.map(totals).fillna(0)
assert not over.any()
print('2. No component count is larger than its release total')

done = ctx.done
latest = fact.apply(lambda r: done[(done.vendor == r.vendor) & (done.window_end < r.window_start)].window_end.max(), axis=1)
assert (latest.isna() | (latest < fact.window_start)).all()
print("3. Every history feature uses only releases that finished before the row's release started")

def density_range(r):
    pm = done[(done.vendor == r.vendor) & (done.window_end < r.window_start) & done.in_model]
    d = [ctx.count_of.get((k, r.component), 0) / p * 10 for k, p in zip(pm.release_key, pm.person_days)]
    return (min(d), max(d)) if d else (np.nan, np.nan)
rng = fact.apply(density_range, axis=1, result_type='expand')
for h in HALF_LIVES:
    col = fact[f'recent_rate_hl{h}']
    ok = col.isna() | ((col >= rng[0] - 1e-9) & (col <= rng[1] + 1e-9))
    assert ok.all(), f'recency hl{h} outside the earlier range'
print('4. Both recency features sit inside the range of earlier densities')

ok = (fact.priority_weighted >= fact.defects) & (fact.priority_weighted <= 8 * fact.defects)
assert ok.all()
print('5. Priority-weighted defects lie between 1 and 8 times the defect count')

1. Release totals match notebook 06 for all 52 releases in the model
2. No component count is larger than its release total
3. Every history feature uses only releases that finished before the row's release started


4. Both recency features sit inside the range of earlier densities
5. Priority-weighted defects lie between 1 and 8 times the defect count


## 5. Save

In [9]:
with sqlite3.connect(CLEAN) as cc:
    fact.to_sql('fact_component_release', cc, if_exists='replace', index=False)
(ROOT / 'reports').mkdir(exist_ok=True)
fact.to_csv(ROOT / 'reports' / 'fact_component_release.csv', index=False)
print('saved fact_component_release:', fact.shape)
fact.groupby(['split', 'vendor']).agg(releases=('release_key', 'nunique'), rows=('defects', 'size'),
                                      rows_with_defects=('defects', lambda s: int((s > 0).sum())),
                                      defects=('defects', 'sum'))

saved fact_component_release: (541, 32)


releases  rows  rows_with_defects  defects
split vendor                                              
test  Vendor A         6   102                 48      328
      Vendor B         4    40                 14       43
      Vendor C         2    12                  5       70
      Vendor D         3     3                  1        1
      Vendor E         1     4                  0        0
train Vendor A        12   204                135     1276
      Vendor B        10   100                 69      541
      Vendor C        10    60                 37      632
      Vendor E         4    16                 13       32

## What is next

Step 3, notebook 08: the label. Which component-release rows count as **high risk**, using defects per person-day of testing (agreed as the best available measure, since no incident or hotfix data exists).